# 01 — Explore Partial Models

Run each partial model standalone and sanity-check outputs.

**Paper**: Neve-Oz, Sherman & Raveh, *Frontiers in Immunology*, 2024

In [ ]:
import subprocess, json, tempfile
from pathlib import Path

ROOT = Path.cwd().parent.parent.parent  # repo root
print(f"Repo root: {ROOT}")

## 1. Membrane Topography

In [ ]:
with tempfile.TemporaryDirectory() as run_dir:
    result = subprocess.run(
        ["python", "-m", "projects.tcr_signaling.models.membrane_topography",
         "--contact_radius", "1.0", "--patch_size", "2.0", "--run-dir", run_dir],
        cwd=str(ROOT), capture_output=True, text=True
    )
    print("stdout:", result.stdout)
    output = json.loads(Path(run_dir, "out", "topography.json").read_text())
    print("Output:", json.dumps(output, indent=2))

## 2. Kinetic Segregation

In [ ]:
with tempfile.TemporaryDirectory() as run_dir:
    result = subprocess.run(
        ["python", "-m", "projects.tcr_signaling.models.kinetic_segregation",
         "--contact_fraction", "0.2", "--cd45_bulk_density", "300.0",
         "--run-dir", run_dir],
        cwd=str(ROOT), capture_output=True, text=True
    )
    print("stdout:", result.stdout)
    output = json.loads(Path(run_dir, "out", "segregation.json").read_text())
    print("Output:", json.dumps(output, indent=2))

## 3. Lck Activity

In [ ]:
with tempfile.TemporaryDirectory() as run_dir:
    result = subprocess.run(
        ["python", "-m", "projects.tcr_signaling.models.lck_activity",
         "--cd45_boundary_density", "375.0", "--lck_decay_length", "0.07",
         "--lck_activation_rate", "0.5", "--contact_radius", "1.0",
         "--run-dir", run_dir],
        cwd=str(ROOT), capture_output=True, text=True
    )
    print("stdout:", result.stdout)
    output = json.loads(Path(run_dir, "out", "lck_activity.json").read_text())
    print("Output:", json.dumps(output, indent=2))

## 4. TCR Phosphorylation

In [ ]:
with tempfile.TemporaryDirectory() as run_dir:
    result = subprocess.run(
        ["python", "-m", "projects.tcr_signaling.models.tcr_phosphorylation",
         "--mean_lck_activity", "50.0", "--tcr_density", "150.0",
         "--phosphorylation_rate", "0.1", "--dephosphorylation_rate", "0.2",
         "--run-dir", run_dir],
        cwd=str(ROOT), capture_output=True, text=True
    )
    print("stdout:", result.stdout)
    output = json.loads(Path(run_dir, "out", "phosphorylation.json").read_text())
    print("Output:", json.dumps(output, indent=2))

## Validate specs with bayesmm

In [ ]:
import glob
for spec in sorted(glob.glob(str(ROOT / "projects/tcr_signaling/specs/model.*.json"))):
    r = subprocess.run(["bayesmm", "validate", spec], cwd=str(ROOT), capture_output=True, text=True)
    status = "OK" if r.returncode == 0 else "FAIL"
    print(f"{Path(spec).name}: {status}")
    if r.returncode != 0:
        print(f"  stderr: {r.stderr.strip()}")